# Capítulo 11 — Constraints, views e transactions: Cozinhar demais pode estragar o banco de dados

**Livro:** começa na página **406** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(11, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## CHECK, por favor: Adicionando uma CHECK CONSTRAINT · PDF p. 410

#### Exercício: CHECK CONSTRAINT

No `cofrinho` só entram moedas de 0.05, 0.10, 0.25, 0.50 e 1.00. Crie a restrição `moeda_valida` e tente inserir uma moeda de 0.03 (`try/except`).

In [ ]:
%sql ALTER TABLE cofrinho ADD CONSTRAINT moeda_valida CHECK (moeda IN (0.05, 0.10, 0.25, 0.50, 1.00))
try:
    %sql INSERT INTO cofrinho (moeda) VALUES (0.03)
except Exception as erro:
    print("Bloqueado pelo CHECK:", "moeda_valida" in str(erro))
%sql SELECT * FROM cofrinho ORDER BY id

## CHECKando o sexo · PDF p. 411

#### Exercício: CHECK em texto

Garanta que `meus_contatos.sexo` só aceita `M` ou `F` e tente atualizar alguém para `X`.

In [ ]:
%sql ALTER TABLE meus_contatos ADD CONSTRAINT sexo_valido CHECK (sexo IN ('M', 'F'))
try:
    %sql UPDATE meus_contatos SET sexo = 'X' WHERE contato_id = 1
except Exception as erro:
    print("Bloqueado pelo CHECK:", "sexo_valido" in str(erro))

## Criando uma view · PDF p. 414

#### Exercício: CREATE VIEW

Crie a view `desenvolvedores_web` com nome, sobrenome, telefone e email de quem deseja um cargo que começa com `Desenvolvedor`.

In [ ]:
%%sql
CREATE VIEW desenvolvedores_web AS
SELECT mc.primeiro_nome, mc.sobrenome, mc.telefone, mc.email
FROM meus_contatos mc
NATURAL JOIN emprego_desejado ed
WHERE ed.cargo LIKE 'Desenvolvedor%';
SELECT * FROM desenvolvedores_web;

## Visualizando suas views · PDF p. 414

#### Exercício: Listando as views

Liste as views do esquema (`information_schema.views`) com a definição de cada uma.

In [ ]:
%%sql
SELECT table_name AS view, view_definition
FROM information_schema.views WHERE table_schema = current_schema();

## O que sua view está realmente fazendo · PDF p. 415

#### Exercício: A view é uma consulta guardada

Mostre que `SELECT * FROM desenvolvedores_web` é o mesmo que usar a consulta da view como subconsulta no `FROM` (dê um apelido para ela).

In [ ]:
%%sql
SELECT * FROM (
    SELECT mc.primeiro_nome, mc.sobrenome, mc.telefone, mc.email
    FROM meus_contatos mc NATURAL JOIN emprego_desejado ed
    WHERE ed.cargo LIKE 'Desenvolvedor%'
) AS dw;

## Inserindo, atualizando e deletando com views · PDF p. 419

#### Exercício: Inserindo pela view

Crie a view `moedas_grandes` (moedas >= 0.50 do `cofrinho`) e insira uma moeda de 1.00 **pela view**. Confira na tabela.

In [ ]:
%%sql
CREATE VIEW moedas_grandes AS SELECT id, moeda, data_moeda FROM cofrinho WHERE moeda >= 0.50;
INSERT INTO moedas_grandes (moeda) VALUES (1.00);
SELECT * FROM cofrinho ORDER BY id;

## View com CHECK OPTION · PDF p. 422

#### Exercício: WITH CHECK OPTION

Sem proteção, a view aceitaria uma moeda de 0.10 (que ela nem mostra). Recrie com `WITH CHECK OPTION` e tente inserir 0.10 por ela (`try/except`).

In [ ]:
%sql CREATE OR REPLACE VIEW moedas_grandes AS SELECT id, moeda, data_moeda FROM cofrinho WHERE moeda >= 0.50 WITH CHECK OPTION
try:
    %sql INSERT INTO moedas_grandes (moeda) VALUES (0.10)
except Exception as erro:
    print("Bloqueado pelo CHECK OPTION:", "check option" in str(erro).lower())

## Quando você tiver terminado com sua view · PDF p. 423

#### Exercício: DROP VIEW

Apague a view `moedas_grandes` e confira a lista de views.

In [ ]:
%%sql
DROP VIEW moedas_grandes;
SELECT table_name FROM information_schema.views WHERE table_schema = current_schema();

## O que aconteceu dentro do caixa eletrônico? · PDF p. 425

#### Exercício: Dinheiro sumindo sem transação

Simule uma transferência de R$ 100 da conta 101 para a 202 que falha no meio: o débito roda, um erro em Python acontece antes do crédito. Mostre o saldo total antes e depois. Em seguida, devolva os R$ 100 à conta 101.

In [ ]:
def total():
    resultado = %sql SELECT SUM(saldo) AS total FROM contas
    return resultado
print("antes:", total())
try:
    %sql UPDATE contas SET saldo = saldo - 100 WHERE conta = 101
    raise RuntimeError("queda de energia no caixa eletrônico!")
    %sql UPDATE contas SET saldo = saldo + 100 WHERE conta = 202
except RuntimeError as erro:
    print(erro)
print("depois:", total())
%sql UPDATE contas SET saldo = saldo + 100 WHERE conta = 101

## O teste clássico ACID · PDF p. 428

#### Exercício: Atomicidade: tudo ou nada

Crie a regra `saldo >= 0` em `contas`. Em Python, com `with motor.begin() as con:` (transação do SQLAlchemy), tente transferir R$ 10.000 da conta 202 para a 101: o crédito roda primeiro e o débito viola a regra. Mostre que **nenhum** dos dois ficou gravado.

In [ ]:
from sqlalchemy import text
%sql ALTER TABLE contas ADD CONSTRAINT saldo_nao_negativo CHECK (saldo >= 0)
try:
    with motor.begin() as con:
        con.execute(text("UPDATE contas SET saldo = saldo + 10000 WHERE conta = 101"))
        con.execute(text("UPDATE contas SET saldo = saldo - 10000 WHERE conta = 202"))
except Exception as erro:
    print("Transação desfeita por inteiro:", type(erro).__name__)
%sql SELECT * FROM contas ORDER BY conta

## SQL ajuda a gerenciar suas transações · PDF p. 429

#### Exercício: BEGIN e COMMIT

Faça a transferência de R$ 100 dentro de uma transação: `BEGIN;`, os dois `UPDATE`s, `COMMIT;`. Mostre os saldos. (No MySQL é `START TRANSACTION`.)

In [ ]:
%%sql
BEGIN;
UPDATE contas SET saldo = saldo - 100 WHERE conta = 101;
UPDATE contas SET saldo = saldo + 100 WHERE conta = 202;
COMMIT;
SELECT * FROM contas ORDER BY conta;

## Agora tente você mesmo · PDF p. 432

#### Exercício: ROLLBACK

Abra uma transação, zere o saldo de todo mundo, confira dentro da própria transação e desfaça com `ROLLBACK`. Termine mostrando os saldos.

In [ ]:
%%sql
BEGIN;
UPDATE contas SET saldo = 0;
ROLLBACK;
SELECT * FROM contas ORDER BY conta;